In [0]:
# =============================================================================
# F1-Pulse | Gold Layer — Driver Performance Analytics
# Notebook: 03_Gold_Analytics
# Author:   Jafar891
# Updated:  2025
#
# Reads Silver enriched_laps, produces aggregated driver performance metrics,
# a constructor standings table, and a lap consistency table.
# All Gold tables are dashboard-ready and saved as Delta.
# Idempotent — safe to re-run.
# =============================================================================

import logging
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, avg, min, max, count, round as spark_round,
    stddev, current_timestamp, lit, rank, dense_rank, percentile_approx
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
import sys
sys.path.append("/Workspace/Users/your-databricks-email/F1-Pulse")
from config.config import (
    YEAR, CATALOG, SILVER, GOLD
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.gold")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def read_silver(table_name: str) -> DataFrame:
    full_table = f"{CATALOG}.{SILVER}.{table_name}"
    try:
        df = spark.read.table(full_table)   # noqa: F821
        count_ = df.count()
        log.info(f"  📥 Read {full_table}  ({count_:,} rows)")
        if count_ == 0:
            raise ValueError(f"Silver table '{full_table}' is empty — cannot produce Gold metrics.")
        return df
    except Exception as e:
        log.error(f"  ❌ Failed to read {full_table}: {e}")
        raise


def write_gold(df: DataFrame, table_name: str) -> None:
    full_table = f"{CATALOG}.{GOLD}.{table_name}"
    try:
        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(full_table)
        )
        log.info(f"  ✅ Written → {full_table}  ({df.count():,} rows)")
    except Exception as e:
        log.error(f"  ❌ Failed to write {full_table}: {e}")
        raise


# ---------------------------------------------------------------------------
# Gold Table 1: Driver Performance Leaderboard
# ---------------------------------------------------------------------------

def build_driver_leaderboard(laps: DataFrame) -> DataFrame:
    """
    Aggregated per-driver metrics across all valid race laps.

    Metrics:
      - fastest_lap_s       : best single lap (seconds, 3dp)
      - avg_lap_time_s      : mean lap time across all valid laps (3dp)
      - median_lap_time_s   : median — more robust to outliers than mean
      - lap_consistency_s   : std deviation of lap times (lower = more consistent)
      - total_valid_laps    : count of laps used in aggregation
      - position_rank       : ranked by fastest lap (1 = fastest overall)
    """
    log.info("  Building driver leaderboard …")

    # Use the is_valid_lap flag set in Silver — no re-filtering needed
    valid_laps = laps.filter(col("is_valid_lap") == True)

    agg = (
        valid_laps
        .groupBy("driver_number", "full_name", "team_name")
        .agg(
            spark_round(min("lap_duration"), 3).alias("fastest_lap_s"),
            spark_round(avg("lap_duration"), 3).alias("avg_lap_time_s"),
            spark_round(
                percentile_approx("lap_duration", 0.5), 3
            ).alias("median_lap_time_s"),
            spark_round(
                stddev("lap_duration"), 3
            ).alias("lap_consistency_s"),       # lower = more consistent
            count("lap_number").cast(IntegerType()).alias("total_valid_laps"),
        )
    )

    # Add ranking window — ordered by fastest lap ascending
    rank_window = Window.orderBy(col("fastest_lap_s").asc())
    agg = agg.withColumn("position_rank", dense_rank().over(rank_window))

    result = (
        agg
        .select(
            "position_rank",
            "full_name",
            "team_name",
            "driver_number",
            "fastest_lap_s",
            "avg_lap_time_s",
            "median_lap_time_s",
            "lap_consistency_s",
            "total_valid_laps",
            current_timestamp().alias("generated_at"),
            lit(YEAR).cast(IntegerType()).alias("season_year"),
        )
        .orderBy("position_rank")
    )

    return result


# ---------------------------------------------------------------------------
# Gold Table 2: Constructor Standings
# ---------------------------------------------------------------------------

def build_constructor_standings(laps: DataFrame) -> DataFrame:
    """
    Team-level summary — aggregates both drivers per constructor.

    Metrics:
      - best_lap_s          : fastest lap from either driver on the team
      - avg_team_pace_s     : mean lap time across all team drivers
      - total_valid_laps    : total laps completed by the team
      - constructor_rank    : ranked by best_lap_s
    """
    log.info("  Building constructor standings …")

    valid_laps = laps.filter(col("is_valid_lap") == True)

    agg = (
        valid_laps
        .groupBy("team_name")
        .agg(
            spark_round(min("lap_duration"), 3).alias("best_lap_s"),
            spark_round(avg("lap_duration"), 3).alias("avg_team_pace_s"),
            count("lap_number").cast(IntegerType()).alias("total_valid_laps"),
        )
    )

    rank_window = Window.orderBy(col("best_lap_s").asc())
    result = (
        agg
        .withColumn("constructor_rank", dense_rank().over(rank_window))
        .select(
            "constructor_rank",
            "team_name",
            "best_lap_s",
            "avg_team_pace_s",
            "total_valid_laps",
            current_timestamp().alias("generated_at"),
            lit(YEAR).cast(IntegerType()).alias("season_year"),
        )
        .orderBy("constructor_rank")
    )

    return result


# ---------------------------------------------------------------------------
# Gold Table 3: Lap-by-Lap Progression (for time-series charts)
# ---------------------------------------------------------------------------

def build_lap_progression(laps: DataFrame) -> DataFrame:
    """
    Lap-level detail for charting driver pace evolution over the race.
    Includes a rolling 5-lap average for smoothed trend lines.

    Keeps only valid laps. One row per driver per lap.
    """
    log.info("  Building lap progression (time-series) …")

    valid_laps = laps.filter(col("is_valid_lap") == True)

    # Rolling 5-lap average per driver, ordered by lap number
    lap_window = (
        Window
        .partitionBy("driver_number")
        .orderBy("lap_number")
        .rowsBetween(-4, 0)   # current + 4 preceding laps
    )

    result = (
        valid_laps
        .withColumn(
            "rolling_avg_5lap_s",
            spark_round(avg("lap_duration").over(lap_window), 3)
        )
        .select(
            "full_name",
            "team_name",
            "driver_number",
            col("lap_number").cast(IntegerType()),
            spark_round(col("lap_duration"), 3).alias("lap_duration_s"),
            "rolling_avg_5lap_s",
            current_timestamp().alias("generated_at"),
            lit(YEAR).cast(IntegerType()).alias("season_year"),
        )
        .orderBy("full_name", "lap_number")
    )

    return result


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def run_gold() -> None:
    log.info("=" * 60)
    log.info(f"F1-Pulse Gold Analytics — {YEAR}")
    log.info("=" * 60)

    results = {"success": [], "failed": []}

    # Read Silver once — reused across all three Gold tables
    log.info("\n[0/3] Reading Silver layer …")
    silver_laps = read_silver("enriched_laps")

    valid_count = silver_laps.filter(col("is_valid_lap") == True).count()
    flagged_count = silver_laps.filter(col("is_valid_lap") == False).count()
    log.info(f"  ✅ Valid laps: {valid_count:,}")
    log.info(f"  ⚠️  Flagged laps (excluded from Gold): {flagged_count:,}")

    # ------------------------------------------------------------------
    # Table 1: Driver Leaderboard
    # ------------------------------------------------------------------
    log.info("\n[1/3] Driver performance leaderboard …")
    try:
        leaderboard = build_driver_leaderboard(silver_laps)
        write_gold(leaderboard, "driver_performance_metrics")
        results["success"].append("driver_performance_metrics")
    except Exception as e:
        log.error(f"  ❌ {e}")
        results["failed"].append("driver_performance_metrics")

    # ------------------------------------------------------------------
    # Table 2: Constructor Standings
    # ------------------------------------------------------------------
    log.info("\n[2/3] Constructor standings …")
    try:
        constructors = build_constructor_standings(silver_laps)
        write_gold(constructors, "constructor_standings")
        results["success"].append("constructor_standings")
    except Exception as e:
        log.error(f"  ❌ {e}")
        results["failed"].append("constructor_standings")

    # ------------------------------------------------------------------
    # Table 3: Lap Progression
    # ------------------------------------------------------------------
    log.info("\n[3/3] Lap-by-lap progression …")
    try:
        progression = build_lap_progression(silver_laps)
        write_gold(progression, "lap_progression")
        results["success"].append("lap_progression")
    except Exception as e:
        log.error(f"  ❌ {e}")
        results["failed"].append("lap_progression")

    # ------------------------------------------------------------------
    # Summary + Display
    # ------------------------------------------------------------------
    log.info("\n" + "=" * 60)
    log.info("GOLD SUMMARY")
    log.info("=" * 60)
    for t in results["success"]:
        log.info(f"  ✅ {CATALOG}.{GOLD}.{t}")
    for t in results["failed"]:
        log.warning(f"  ❌ {t} — check logs above")
    log.info("=" * 60)

    # Preview all Gold tables in the notebook output
    if "driver_performance_metrics" in results["success"]:
        log.info("\n🏎️  Driver Leaderboard — 2025 Abu Dhabi Grand Prix")
        display(spark.table(f"{CATALOG}.{GOLD}.driver_performance_metrics"))  # noqa: F821

    if "constructor_standings" in results["success"]:
        log.info("\n🏭  Constructor Standings")
        display(spark.table(f"{CATALOG}.{GOLD}.constructor_standings"))       # noqa: F821

    if "constructor_standings" in results["success"]:
        log.info("\n📈  Lap Progression (first 10 rows)")
        display(                                                               # noqa: F821
            spark.table(f"{CATALOG}.{GOLD}.lap_progression").limit(20)
        )

    if results["failed"]:
        raise RuntimeError(f"Gold layer completed with errors: {results['failed']}")


run_gold()